In [0]:
from pyspark.sql.functions import current_timestamp, col

# Leitura do arquivo TSV (tratado como CSV com delimitador TAB)
df = (
    spark.read
        .format("csv")                              # Define o formato como CSV
        .option("header", "true")                   # Primeira linha contém o cabeçalho
        .option("delimiter", "\t")                  # Delimitador TAB (TSV)
        .option("inferSchema", "true")               # Infere automaticamente o schema
        .load("/Volumes/techdados/bronze/vol_landing/orders.tsv")  # Caminho do arquivo
        .withColumn(
            "_source_file",                          # Coluna técnica de rastreabilidade
            col("_metadata.file_path")               # Caminho do arquivo de origem
        )
        .withColumn(
            "_ingestion_timestamp",                  # Coluna técnica de auditoria
            current_timestamp()                      # Timestamp da ingestão
        )
)

# Escrita do DataFrame como tabela Delta na camada Bronze
(
    df.write
        .format("delta")                             # Formato Delta Lake
        .mode("overwrite")                           # Sobrescreve a tabela se existir
        .option("overwriteSchema", "true")           # Permite atualizar o schema
        .saveAsTable("techdados.bronze.orders")                # Nome da tabela no metastore
)


In [0]:
%sql
CREATE OR REPLACE TABLE techdados.bronze.orders
USING DELTA
AS
SELECT
    *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingestion_timestamp
FROM read_files(
    '/Volumes/techdados/bronze/vol_landing/orders.tsv',
    format => 'csv',
    header => true,
    delimiter => '\t',
    inferSchema => true
);


In [0]:
%sql
select * from techdados.bronze.orders